# Chapter 3: Giving Agents Hands

This notebook mirrors the chapter. Run the cells in order. Nothing here needs an
API key: every cell that involves a model drives the real SDK runner through its
scripted test model, which is also how the chapter's three failure drills were
captured. Pinned versions: openai-agents 0.24.0, mcp 2.3.0.

In [1]:
from importlib.metadata import version
print("openai-agents", version("openai-agents"), "| mcp", version("mcp"))

openai-agents 0.24.0 | mcp 2.3.0


## Designing tools the model can actually use

Three intent-shaped ERP tools. Look at the model-facing schemas: the
`idempotency_key` is nowhere in them, because it travels in the run context.

In [2]:
from examples import intent_tools
intent_tools.main()

preview_po_change
  description: Show what a purchase order change would do without making it.
  parameters : proposal
  schema     :
    {
      "$defs": {
        "POChangeProposal": {
          "properties": {
            "po_number": {
              "description": "Purchase order identifier",
              "title": "Po Number",
              "type": "string"
            },
            "line_item": {
              "description": "Line item position",
              "minimum": 1,
              "title": "Line Item",
              "type": "integer"
            },
            "new_quantity": {
              "minimum": 0,
              "title": "New Quantity",
              "type": "integer"
            },
            "reason_code": {
              "enum": [
                "price_correction",
                "vendor_change",
                "cancel_line"
              ],
              "title": "Reason Code",
              "type": "string"
            }
          },
          "required": 

## Engineering the side-effect boundary

### Values the model must not control travel in the run context

In [3]:
from examples import run_context
await run_context.main()

model arguments : {"customer_id": "cus_8841", "original_charge_id": "ch_3f9a21", "amount_cents": 4500, "reason": "Duplicate order shipped"}
run context     : idempotency_key=fedf5a3747f8bd6e220389195030b5e9 actor_id=agent.m.smith@example.com
tool result     : status='issued' refund_id='rf_1c2d3e' amount_cents=4500
context fields in the model's arguments: none
final output    : Refund rf_1c2d3e for $45.00 has been issued.


### Drill 1: retry after a lost response

The orchestration layer derives a key, the first response is lost, the retry
reuses the key, and the store ends up with exactly one refund.

In [4]:
from drills import retry_after_timeout
await retry_after_timeout.main()

first call : issued   refund_id=rf_7f3a9c amount_cents=21450
network    : response lost, orchestration retries with the same key
second call: replayed refund_id=rf_7f3a9c amount_cents=21450
refunds in store: 1


### Preview and execute, bound by a content hash

In [5]:
from examples import preview_execute
await preview_execute.main()

preview        : amount_cents=4500 content_hash=347bcee3c482b0b4
execute (match): status=issued refund_id=rf_1c2d3e
execute (stale): refused error_class=preview_mismatch retryable=False hint=replan
refunds in store: 1


### Drill 2: a rejected approval

`execute_refund` is gated by a policy function. The run stops with one
interruption, a scripted reviewer says no, and the tool never runs.

In [6]:
from drills import reject_approval
await reject_approval.main()

interruption : execute_refund
               customer_id="cus_8841"
               original_charge_id="ch_3f9a21"
               amount_cents=21450
               reason="Damaged on arrival"
reviewer     : rejected
reason       : Amount exceeds the self-service limit for this customer tier
tool ran     : no (refunds in store: 0)
final output : I was not able to issue the $214.50 refund. A reviewer
               declined it because the amount exceeds the self-service
               limit for this customer tier. I can raise it with the
               billing team if you would like.


### Resuming a parked run from a serialized state, in another process

In [7]:
from examples import resume_from_string
await resume_from_string.main()

RunState context was serialized from a Pydantic model. Provide context_deserializer or context_override to restore the original type.


process A : paused on execute_refund, state persisted (10340 bytes)
process A : key in context 110ffd1855f17c477cad725406e3dd31
process B : rebuilt state with context_override, approved, resumed
process B : tool result status='issued' refund_id='rf_7f3a9c' amount_cents=21450
process B : final output The refund was approved and issued.
store     : refund rf_7f3a9c written under key 110ffd1855f17c477cad725406e3dd31
key match : yes


## Plugging in with the Model Context Protocol

### Drill 3: the server is not there

Nothing is listening on port 8765, so the connection fails fast, before any
model call, with an error that names the server.

In [8]:
from drills import unavailable_mcp_server
await unavailable_mcp_server.main()

error  : Failed to connect to MCP server 'finance-evidence'
reason : Could not reach the server. (http://127.0.0.1:8765/mcp)
model calls made: 0


### Connecting to the finance evidence server

Start the server in a terminal first:

```
python -m mcp_servers.finance_evidence_server
```

Then run the next cell. The tool list is fetched when the run starts, inside the
`async with` block, and a run outside the block fails with `Server not initialized`.

In [9]:
from examples import mcp_connect
await mcp_connect.main()

connected  : finance-evidence
run started: 1 tool(s) listed for the model: get_trial_balance_snapshot
tool result: { "ok": true, "entity": "UK01", "period": "2026-Q3", "materiality_threshold": 1000000, "source_ref": "tb:UK01:...
final      : Six accounts sit above the threshold for UK01 in 2026-Q3.
resource   : schema://accounts
             accounts:
               - code: "1000"
                 name: Cash and cash equivalents
after block: Server not initialized, model calls made: 0
OPENAI_API_KEY is not set; the live question was skipped


## Building a financial services audit-prep agent

The complete flow: reads from the data and policy servers, draft artifacts written
under idempotency keys, one interruption at `publish_evidence_bundle`, approval,
resume, and a tool-call journal row for every call with the principal, the
argument hash, and the policy version.

In [10]:
from audit_prep import main as audit_main
await audit_main.main()

run        : run_ae_0007 principal=control.owner@example.com tenant=acme-uk
server     : finance-data (3 tools)
server     : reporting-policy (1 tool)
server     : evidence-store (3 tools)
tool calls : get_trial_balance_snapshot, get_journal_entries, get_reporting_requirements, preview_evidence_bundle, write_evidence_artifact, write_evidence_artifact, write_evidence_artifact, publish_evidence_bundle
interrupt  : publish_evidence_bundle(bundle_id=eb_d5d9f967, content_hash=20452e8046f3834f)
reviewer   : approved
final      : bundle eb_d5d9f967 published, 3 artifacts
journal    : 8 rows
  get_trial_balance_snapshot  succeeded principal=control.owner@example.com args=8ab88a0b policy=unknown
  get_journal_entries         succeeded principal=control.owner@example.com args=2f17e5c8 policy=unknown
  get_reporting_requirements  succeeded principal=control.owner@example.com args=7402e5f8 policy=rules-2026.07
  preview_evidence_bundle     succeeded principal=control.owner@example.com args=5100295

## Run-and-verify

Table 3.1 in the chapter lists what each command should show. The test suite
covers the same ground:

```
pytest -q
```